# MAI-Transcribe-2: turn A Bit of AI into a readable episode

Henk and Amy have recorded an episode of **A Bit of AI**. Now the editor
needs to find who said what, check the show's names and AI terminology,
and prepare a readable transcript for listeners.

This approximately 8-10 minute demo follows that recording through one
production workflow: **listen -> transcribe -> follow the hosts -> revisit
a moment -> prepare readable text -> check terminology -> include Dutch
listeners -> hand off the transcript**.

The [MAI-Voice notebook](../MAI-Voice-2.1/demo.ipynb) gives the podcast a
voice. This notebook goes the other way: **recorded speech becomes text
and navigable metadata**. Neither notebook requires running the other.

**Before running:** follow the [setup instructions](demo-instructions.md)
and supply a permitted podcast excerpt. No recordings or credentials are
bundled. A 30-90 second, two-speaker mono PCM WAV works well; MP3 and FLAC
also work for transcription, but not the word-excerpt player.

**Cost and privacy:** The three main transcription cells each make one paid
request. The optional Dutch cell makes one more. Run cells individually:
**Run All attempts all four and writes the local exports if it completes**;
a missing Dutch file stops execution. There are no retries or substitutions.
Audio is sent to Azure. Clear outputs before sharing: players embed audio,
and tables retain speech-derived text.

This is completed-file transcription using **MAI-Transcribe-2**, not live
streaming, translation, or automatic show-note generation.
[Extended model information](information.md).

## 1. Set up the A Bit of AI editing desk

In Codespaces, select **Python (MAI demos)** and supply `SPEECH_ENDPOINT` and
`SPEECH_KEY` through [Codespaces secrets](demo-instructions.md#github-codespaces).
Locally, use your `.venv` kernel and put those values in `.env` beside this notebook.
Never paste a key into a cell.

Authentication is automatic: a configured resource key is used; otherwise
the helper uses Azure CLI credentials after `az login`. Your endpoint must be
the HTTPS resource root and your identity must have the required Speech role.

The only recording path needed now is `AUDIO_PATH`. `AUDIO_FILE` in `.env`
still works, including for an existing recording with a different filename.
The Dutch path and keyword list live beside their examples.

Rerun the next cell after editing `.env`; no restart is needed. Settings are
read fresh from this directory, with process environment variables taking
precedence. Changed Codespaces secrets require restarting the codespace.

There are no run or export switches: **run the cells you want and skip the
optional ones**. Each transcription cell sends audio to Azure; the final export
cell writes local files. `samples/`, WAV files, and `outputs/` stay Git-ignored.

In [ ]:
import copy
import io
import json
import math
import os
import time
import wave
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import pandas as pd
import requests
from dotenv import dotenv_values
from IPython.display import Audio, FileLink, display


In [ ]:
SETTINGS = {**dotenv_values(Path.cwd() / ".env"), **os.environ}
SPEECH_ENDPOINT = (SETTINGS.get("SPEECH_ENDPOINT") or "").strip()
AUDIO_PATH = Path(SETTINGS.get("AUDIO_FILE") or "samples/a-bit-of-ai-en.wav")

print("Podcast recording:", AUDIO_PATH)

### API plumbing

The helper fixes the model, API version, and demo size limit so they do not
clutter setup. The actual transcription options remain visible in the demo.
It validates inputs before contacting Azure, sets timeouts, rejects redirects,
and reports errors without printing credentials or raw service error bodies.

The endpoint must be a public Azure `cognitiveservices.azure.com` resource
root, not a Foundry project URL. Sovereign-cloud endpoints require a reviewed
configuration change.

In [ ]:
MODEL = "MAI-Transcribe-2"
API_VERSION = "2025-10-15"
MAX_DEMO_BYTES = 20 * 1024 * 1024  # Fixed demo limit, not a service quota.
_credential = None


def resource_endpoint(value):
    parsed = urlsplit(value)
    hostname = parsed.hostname or ""
    suffix = ".cognitiveservices.azure.com"
    if (parsed.scheme != "https" or not hostname.endswith(suffix)
            or not hostname[:-len(suffix)] or parsed.username or parsed.password
            or parsed.port not in (None, 443) or parsed.path not in ("", "/")
            or parsed.query or parsed.fragment):
        raise ValueError("Set SPEECH_ENDPOINT to https://<resource>.cognitiveservices.azure.com")
    return value.rstrip("/")


def validate_audio(path):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Recording not found: {path}. Add your audio and update AUDIO_PATH.")
    if path.suffix.lower() not in {".wav", ".mp3", ".flac"}:
        raise ValueError("Use a WAV, MP3, or FLAC recording.")
    if not 0 < path.stat().st_size <= MAX_DEMO_BYTES:
        raise ValueError("Use a nonempty recording no larger than the demo's 20 MiB limit.")
    if path.suffix.lower() == ".wav":
        try:
            with wave.open(str(path), "rb") as recording:
                if recording.getnchannels() != 1 or recording.getsampwidth() != 2:
                    raise ValueError("For this demo, use a mono, 16-bit PCM WAV recording.")
                if recording.getnframes() == 0:
                    raise ValueError("The WAV recording has no audio frames.")
        except (wave.Error, EOFError) as exc:
            raise ValueError("Cannot read this WAV. Export it as mono, 16-bit PCM WAV.") from exc
    return path


def auth_headers():
    global _credential
    key = (SETTINGS.get("SPEECH_KEY") or "").strip()
    if key:
        return {"Ocp-Apim-Subscription-Key": key}
    from azure.identity import AzureCliCredential
    if _credential is None:
        _credential = AzureCliCredential()
    token = _credential.get_token("https://cognitiveservices.azure.com/.default")
    return {"Authorization": f"Bearer {token.token}"}


def request_definition(style="verbatim", phrases=None, diarization=True):
    if style not in {"verbatim", "clean"}:
        raise ValueError("Style must be 'verbatim' or 'clean'.")
    definition = {
        "enhancedMode": {
            "enabled": True,
            "model": MODEL,
            "modelOptions": {"timestamps": "word", "transcribeStyle": style},
        },
        "diarization": {"enabled": diarization},
    }
    if phrases:
        if not all(isinstance(term, str) and term.strip() for term in phrases):
            raise ValueError("Keyword hints must be nonempty strings.")
        definition["phraseList"] = {"phrases": list(phrases)}
    return definition


def validate_result(payload):
    if not isinstance(payload, dict):
        raise ValueError("Expected a JSON object from the transcription service.")
    combined = payload.get("combinedPhrases")
    if not isinstance(combined, list) or not all(
        isinstance(item, dict) and isinstance(item.get("text"), str) for item in combined
    ):
        raise ValueError("Response is missing a valid combinedPhrases transcript list.")
    segments = payload.get("phrases", [])
    if not isinstance(segments, list):
        raise ValueError("Response phrases must be a list.")
    duration = payload.get("durationMilliseconds")
    if duration is not None and not valid_time(duration):
        raise ValueError("Response audio duration must be a nonnegative finite number.")
    for segment in segments:
        if not isinstance(segment, dict) or not isinstance(segment.get("text"), str):
            raise ValueError("A transcript segment is missing text.")
        words = segment.get("words", [])
        if not isinstance(words, list) or not all(
            isinstance(word, dict) and isinstance(word.get("text"), str) for word in words
        ):
            raise ValueError("A transcript segment contains invalid word metadata.")
        for timed_item in [segment, *words]:
            for field in ("offsetMilliseconds", "durationMilliseconds"):
                if field in timed_item and not valid_time(timed_item[field]):
                    raise ValueError(f"Invalid {field} in transcript metadata.")
    return payload


def valid_time(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value) and value >= 0)


@dataclass
class TranscriptionRun:
    audio_path: Path
    definition: dict
    response: dict
    elapsed_seconds: float

    @property
    def text(self):
        return "\n".join(item["text"] for item in self.response["combinedPhrases"])


def transcribe(path, definition):
    endpoint = resource_endpoint(SPEECH_ENDPOINT)
    path = validate_audio(path)
    headers = auth_headers()
    url = f"{endpoint}/speechtotext/transcriptions:transcribe"
    started = time.perf_counter()
    try:
        with path.open("rb") as audio:
            with requests.post(
                url, params={"api-version": API_VERSION}, headers=headers,
                files={"audio": (path.name, audio, "application/octet-stream")},
                data={"definition": json.dumps(definition)},
                timeout=(10, 180), allow_redirects=False,
            ) as response:
                if response.status_code != 200:
                    guidance = {
                        400: "Check the audio format and model-specific request options.",
                        401: "Check the resource key or sign in again with az login.",
                        403: "Check resource permissions and model access.",
                        404: "Check the resource endpoint, region, and model availability.",
                        413: "The service rejected the recording size; use a shorter clip.",
                        429: "The resource is throttled; wait before manually retrying.",
                    }.get(response.status_code, "Check service health and endpoint configuration before retrying.")
                    raise RuntimeError(f"Azure transcription returned HTTP {response.status_code}. {guidance} No automatic retry was made.")
                try:
                    payload = response.json()
                except ValueError:
                    raise RuntimeError("Azure returned HTTP 200 but the response was not valid JSON.") from None
    except requests.RequestException as exc:
        raise RuntimeError(f"Transcription request failed ({type(exc).__name__}). Check connectivity. No automatic retry was made; a timed-out request may still have been billed.") from None
    return TranscriptionRun(path, copy.deepcopy(definition), validate_result(payload), time.perf_counter() - started)


def transcript_tables(run):
    segment_rows, word_rows = [], []
    for index, segment in enumerate(run.response.get("phrases", [])):
        identity = {"segment": index, "speaker": segment.get("speaker"), "language": segment.get("locale")}
        def row(item):
            start = item.get("offsetMilliseconds")
            duration = item.get("durationMilliseconds")
            return {**identity, "start_s": None if start is None else start / 1000,
                    "duration_s": None if duration is None else duration / 1000,
                    "text": item["text"]}
        segment_rows.append(row(segment))
        word_rows.extend(row(word) for word in segment.get("words", []))
    columns = ["segment", "speaker", "language", "start_s", "duration_s", "text"]
    return pd.DataFrame(segment_rows, columns=columns), pd.DataFrame(word_rows, columns=columns)


def show_run(run):
    duration = run.response.get("durationMilliseconds")
    audio_seconds = None if duration is None else duration / 1000
    display(pd.DataFrame([{
        "model": run.definition["enhancedMode"]["model"],
        "audio_seconds": audio_seconds,
        "request_seconds": round(run.elapsed_seconds, 3),
    }]))
    print(run.text if run.text.strip() else "No speech was returned by the service.")


def wav_excerpt(path, start_seconds, end_seconds):
    path = validate_audio(path)
    if path.suffix.lower() != ".wav":
        raise ValueError("Word playback requires a mono PCM WAV file. MP3/FLAC still support transcription tables.")
    if not valid_time(start_seconds) or not valid_time(end_seconds) or end_seconds <= start_seconds:
        raise ValueError("Choose a finite playback interval with 0 <= start < end.")
    with wave.open(str(path), "rb") as source:
        rate = source.getframerate()
        start_frame = int(start_seconds * rate)
        end_frame = min(source.getnframes(), int(end_seconds * rate))
        if start_frame >= end_frame:
            raise ValueError("Playback interval falls outside this recording.")
        source.setpos(start_frame)
        frames = source.readframes(end_frame - start_frame)
        buffer = io.BytesIO()
        with wave.open(buffer, "wb") as output:
            output.setnchannels(source.getnchannels())
            output.setsampwidth(source.getsampwidth())
            output.setframerate(rate)
            output.writeframes(frames)
    return buffer.getvalue()


def export_run(run, root=Path("outputs")):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    directory = root / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
    directory.mkdir()
    (directory / "transcript.txt").write_text(run.text, encoding="utf-8")
    (directory / "response.json").write_text(json.dumps(run.response, ensure_ascii=False, indent=2), encoding="utf-8")
    metadata = {"model": run.definition["enhancedMode"]["model"], "request_definition": run.definition,
                "elapsed_seconds": run.elapsed_seconds, "exported_at": datetime.now(timezone.utc).isoformat()}
    (directory / "run.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return directory


## 2. The episode is recorded: listen to Henk and Amy

Start with a short A Bit of AI conversation: the welcome, a topic teaser,
and a natural hesitation or correction. Use your own permitted episode
excerpt or the [fictional rehearsal script](demo-instructions.md#prepare-the-recordings).

Listen for three things we will revisit: **the host handoffs, a filler or
correction, and the show/product names**. Those give the transcript,
timestamps, style comparison, and keyword hints a shared purpose.

This cell checks the endpoint and local file but makes **no Azure calls**.
The player embeds the recording in notebook output; clear it before sharing.

In [ ]:
_ = resource_endpoint(SPEECH_ENDPOINT)
_ = validate_audio(AUDIO_PATH)
display(Audio(filename=str(AUDIO_PATH.expanduser())))


## 3. Make the first transcript of the episode

Before editing anything, capture a **verbatim** baseline of the conversation.
The visible request selects MAI-Transcribe-2, speaker diarization, and
word timestamps. No language is forced and no keyword hints are supplied.

Read the result as an editor: did it capture the welcome, the conversation,
and the correction you just heard? The recording remains the source of truth.

**Running the next cell uploads the recording to Azure and incurs usage
charges.** Request time includes upload and response handling, not credential
acquisition. The service's audio duration is a separate measurement.

In [ ]:
baseline_definition = request_definition()
print(json.dumps(baseline_definition, indent=2))
baseline = None  # Do not leave an older result looking current if a rerun fails.
baseline = transcribe(AUDIO_PATH, baseline_definition)
show_run(baseline)


## 4. Follow the conversation between the hosts

A podcast transcript needs more than one block of text. Inspect the segment
table to see where the speaker changes, then compare those turns with the
recording.

Speaker IDs are **anonymous labels, not Henk/Amy identities**. Listen before
assigning a host name; the notebook does not infer a name from a speaker
number. Missing speaker/timing fields stay missing. Confidence is not
presented as an accuracy percentage.

Full segment and word tables remain available as `segments` and `words`.

In [ ]:
segments, words = transcript_tables(baseline)
print(f"{len(segments)} segments; {len(words)} timed-word entries")
if segments.empty:
    print("No segment metadata was returned for this request.")
else:
    display(segments.head(30))
if words.empty:
    print("No word metadata was returned. Check the model options and actual response.")
else:
    display(words.head(40))


### Find the moment the editor wants to check

Pick a word near a host handoff, a name, or the correction you heard.
The slider plays that word with 0.35 seconds of context on each side.
This is how an editor can check a line without replaying the whole episode.

Playback uses the returned timestamps and your local PCM WAV:
**no new Azure request**. Press Play in the audio control.

Widgets need JupyterLab widget support or VS Code's Jupyter extension.
If unavailable, use the timestamp table and call
`display(Audio(data=wav_excerpt(AUDIO_PATH, 1.0, 3.0)))` with your chosen
start/end seconds.

In [ ]:
if baseline.audio_path.suffix.lower() != ".wav":
    print("Word playback is WAV-only; use the timestamp tables for this MP3/FLAC recording.")
elif words.dropna(subset=["start_s", "duration_s"]).empty:
    print("Word playback unavailable: the response has no usable word timings.")
else:
    import ipywidgets as widgets
    timed_words = words.dropna(subset=["start_s", "duration_s"]).reset_index(drop=True)
    def play_word(index):
        word = timed_words.iloc[index]
        start = max(0.0, float(word.start_s) - 0.35)
        end = float(word.start_s + word.duration_s) + 0.35
        print(f"{word.text} | speaker {word.speaker} | starts at {word.start_s:.3f}s")
        display(Audio(data=wav_excerpt(baseline.audio_path, start, end)))
    word_picker = widgets.IntSlider(value=0, min=0, max=len(timed_words) - 1,
                                    description="Word index", continuous_update=False)
    display(widgets.VBox([word_picker, widgets.interactive_output(play_word, {"index": word_picker})]))


## 5. Prepare a readable transcript for the episode page

Keep the verbatim baseline for checking what was said. Now change only
`transcribeStyle` to **clean** on the same recording and compare a more
readable draft for the A Bit of AI episode page.

Look at the hesitation or correction you heard earlier. Clean output can
remove verbal clutter, but it is **not a summary or generated show notes**.
Review both versions against the audio before publishing. If the recording
has no disfluencies, the outputs may legitimately be similar.

Running this cell makes **one additional paid request**. Skip it if you only
need the verbatim baseline.

In [ ]:
clean = None
clean_definition = copy.deepcopy(baseline.definition)
clean_definition["enhancedMode"]["modelOptions"]["transcribeStyle"] = "clean"
clean = transcribe(baseline.audio_path, clean_definition)
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"Recording (verbatim)": [baseline.text], "Episode-page draft (clean)": [clean.text]}))
print(f"Clean request: {clean.elapsed_seconds:.3f} seconds")

## 6. Check the show name and AI vocabulary

The episode page should spell **A Bit of AI**, the hosts' names, and the
products discussed correctly. Edit `keywords` in the next cell to contain only terms actually
spoken in your recording; the defaults match the fictional rehearsal script.

Compare the original **verbatim** baseline with another verbatim request
using `phraseList.phrases`. Keep every other option unchanged so this tests
vocabulary hints, not the effect of switching to clean output.

Hints are not search-and-replace rules and do not force words into the
transcript. A correct baseline may not improve. This makes **one additional
paid request** when you run the cell.

In [ ]:
keywords = ["A Bit of AI", "Henk Boelman", "Amy", "Microsoft Foundry", "MAI-Transcribe", "AgentCon"]

hinted = None
if not keywords or not all(isinstance(term, str) and term.strip() for term in keywords):
    raise ValueError("Supply a nonempty list of terms spoken in the recording.")
hinted_definition = copy.deepcopy(baseline.definition)
hinted_definition["phraseList"] = {"phrases": list(keywords)}
print("Keyword hints:", keywords)
hinted = transcribe(baseline.audio_path, hinted_definition)
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"Without hints": [baseline.text], "With hints": [hinted.text]}))
print(f"Keyword-hinted request: {hinted.elapsed_seconds:.3f} seconds")

## 7. Optional: include Henk's Dutch welcome

A Bit of AI also has a Dutch introduction. Listen to a permitted Dutch clip,
then transcribe it without setting a language hint. The output should stay
Dutch: **transcription is not translation**.

Set the path directly in the next cell, or **skip the cell** if you do not
want this example. A missing file raises an error before any upload; it is
not silently skipped. Existing filenames such as `samples/meeting-nl.wav`
can be used by editing the path.

Running the cell makes **one additional paid request**. No `locales` hint
is sent. Dutch is one of 60 documented languages; mixed-language performance
still needs checking on the actual recording.

In [ ]:
dutch = None
dutch_path = validate_audio(Path("samples/a-bit-of-ai-nl.wav"))
display(Audio(filename=str(dutch_path)))
dutch = transcribe(dutch_path, request_definition())
show_run(dutch)
dutch_segments, _ = transcript_tables(dutch)
display(dutch_segments.head(20))

## 8. Hand the transcript to the episode editor

We now have a verbatim record with navigation metadata and, after running the comparison cells,
separate clean, keyword-hinted, and Dutch results to review. These are
material for an episode page or a later show-note workflow, not a finished
summary, caption file, or automatically published page.

`baseline.response` holds the real JSON response and `baseline.definition`
holds the sanitized request. Inspect private speech content only when
appropriate for your audience.

Run the next cell to save **the baseline only**: `transcript.txt`,
`response.json`, and `run.json` in a unique `outputs/` folder. Clean, hinted,
and Dutch results remain in memory; the export does not silently pick a
different version. No audio, credentials, or endpoint is exported.
Transcript text and phrase hints may still be sensitive.

Skip the export cell to keep results only in notebook memory and output.

In [ ]:
export_directory = export_run(baseline)
print("Saved baseline results to", export_directory)
for name in ("transcript.txt", "response.json", "run.json"):
    display(FileLink(str(export_directory / name)))

## A Bit of AI: from recording to editorial handoff

We started with a podcast conversation, not a list of API features:

- The verbatim transcript provides a first record of the episode.
- Anonymous speaker labels help follow the host handoffs.
- Word timestamps let the editor revisit a name or correction.
- Clean output provides a readable transcript draft, not show notes.
- Keyword hints test the spelling of the show, hosts, and AI terminology.
- A Dutch welcome stays Dutch without manually selecting a language.
- Optional baseline TXT/JSON exports give a later workflow real source material.

Review names, numbers, corrections, and speaker attribution against the audio
before publishing. Request timings are observations, not benchmark promises.
Summaries, chapter titles, translations, and action items need separate processing.

**The companion story:** [MAI-Voice](../MAI-Voice-2.1/demo.ipynb) turns a script
into the podcast opening; this notebook turns a recorded episode into text.
They can be demonstrated independently. If using generated audio as input,
disclose that it is synthetic; it is not evidence of accuracy on natural speech.

**Streaming is separate:** MAI-Transcribe-2-Streaming needs its own integration.
Uploading this finished episode recording is not live transcription.

### Troubleshooting

| Problem | Check |
| --- | --- |
| File not found | Notebook working directory, `AUDIO_PATH`, and the sample file name |
| Invalid WAV | Export mono, 16-bit PCM WAV; renaming a compressed file does not convert it |
| `az` / Entra authentication error | Azure CLI installation, `az login`, subscription, and Speech resource permissions; or set `SPEECH_KEY` in `.env` and rerun configuration |
| HTTP 401 / 403 | Matching resource credentials, role assignment, and model access |
| HTTP 400 / 404 | Explicit model selection, API version, supported region, endpoint, and audio format |
| HTTP 429 / timeout | Wait and inspect resource/service health before manually retrying; the original call may still be billable |
| No words or speakers | Inspect the real response and requested options; do not fabricate missing metadata |
| No widget output | Enable widgets or use the plain timestamp table and manual WAV excerpt helper |

**Before sharing:** use Restart Kernel and Clear All Outputs, save the notebook, and inspect your Git diff. Clearing outputs removes embedded audio and transcript displays; restarting the kernel also releases in-memory credentials and results.

### References

- [MAI-Transcribe model guide](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe)
- [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech)
- [Speech role-based access control](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/role-based-access-control)
- [Streaming model overview](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe-2-streaming)
- [Extended model information](information.md)
